```
# Lab type:  prompt
# Course:    NL301 Natural Language Processing with Python
# Lesson:    06 — LLM-Assisted NLP
# Task:      Complete three prompt templates for support-ticket classification
#            using the Anthropic API.
```

## Setup

Requires a free `GOOGLE_API_KEY` from Google AI Studio:

1. Go to <a href="https://aistudio.google.com" target="_blank" rel="noopener noreferrer">aistudio.google.com</a> and sign in with any Google account.
2. Click **Get API key** → **Create API key** and copy the key.
3. **In Colab:** open the Secrets panel (🔑 icon in the left sidebar), add a secret named `GOOGLE_API_KEY`, and paste your key.
   **Locally:** set it in your shell before launching Jupyter: `export GOOGLE_API_KEY="your-key"`.

The Gemini free tier allows up to 15 requests per minute and 1 million tokens per day — enough for all tasks in this lab.

In [ ]:
!pip install google-generativeai --quiet

In [ ]:
import google.generativeai as genai
import json, os
from typing import Optional

try:
    from google.colab import userdata
    _api_key = userdata.get("GOOGLE_API_KEY")
except ImportError:
    _api_key = os.environ["GOOGLE_API_KEY"]

genai.configure(api_key=_api_key)
model = genai.GenerativeModel("gemini-2.0-flash")

# Support ticket dataset
tickets = [
    {"id": 1, "text": "My order arrived damaged and I want a full refund immediately."},
    {"id": 2, "text": "The product exceeded my expectations — absolutely fantastic!"},
    {"id": 3, "text": "Can you tell me whether you ship to Canada?"},
    {"id": 4, "text": "I've been waiting three weeks and still no delivery."},
    {"id": 5, "text": "Do you offer student discounts on annual subscriptions?"},
    {"id": 6, "text": "The app keeps crashing on iOS 17. This is unacceptable."},
    {"id": 7, "text": "Just wanted to say your support team was incredibly helpful."},
    {"id": 8, "text": "Please cancel my subscription effective immediately."},
]
CATEGORIES = ["complaint", "praise", "inquiry"]

---
## Task 1: Zero-shot classification

Complete the `zero_shot_prompt` so that the model returns exactly one word: `complaint`, `praise`, or `inquiry`. No other output.

In [ ]:
def classify_zero_shot(text: str) -> str:
    # TODO: write a prompt that returns exactly one category word.
    # Hint: be explicit about output format — one word, nothing else.
    zero_shot_prompt = f"""
    YOUR PROMPT HERE
    
    Ticket: {text}
    """
    
    response = model.generate_content(zero_shot_prompt)
    return response.text.strip().lower()

# Test on first 4 tickets
for t in tickets[:4]:
    label = classify_zero_shot(t["text"])
    print(f"[{label:10s}]  {t['text'][:60]}")

**Analysis:** Does zero-shot correctly classify the complaint about a damaged order? The cancellation request — is that a complaint or inquiry?

*(Write your answer here.)*

---
## Task 2: Few-shot classification

Add 2 labelled examples per category to your prompt. Then compare zero-shot vs few-shot accuracy on all 8 tickets (use manual ground-truth labels below).

In [ ]:
GROUND_TRUTH = {1: "complaint", 2: "praise", 3: "inquiry", 4: "complaint",
                5: "inquiry",   6: "complaint", 7: "praise", 8: "inquiry"}

def classify_few_shot(text: str) -> str:
    # TODO: build a few-shot prompt with 2 examples per category (6 examples total).
    # Each example: show the ticket text and the correct label.
    few_shot_prompt = f"""
    YOUR FEW-SHOT PROMPT HERE
    
    Ticket: {text}
    """
    
    response = model.generate_content(few_shot_prompt)
    return response.text.strip().lower()

# Evaluate both approaches
def accuracy(fn):
    correct = sum(fn(t["text"]) == GROUND_TRUTH[t["id"]] for t in tickets)
    return correct / len(tickets)

print(f"Zero-shot accuracy: {accuracy(classify_zero_shot):.2f}")
print(f"Few-shot  accuracy: {accuracy(classify_few_shot):.2f}")

**Analysis:** Which approach performed better? When would you expect few-shot to consistently outperform zero-shot?

*(Write your answer here.)*

---
## Task 3: Structured JSON extraction

Write a prompt that extracts four fields from each ticket: `sentiment` (positive/negative/neutral), `issue_type` (billing/shipping/technical/general), `urgency` (low/medium/high), `action_required` (bool). Handle `json.JSONDecodeError`.

In [ ]:
from dataclasses import dataclass

@dataclass
class TicketAnalysis:
    sentiment: str
    issue_type: str
    urgency: str
    action_required: bool

def extract_ticket_info(text: str) -> Optional[TicketAnalysis]:
    # TODO: write a prompt instructing the model to return ONLY valid JSON
    # with exactly these four keys.  Wrap json.loads in a try/except.
    extraction_prompt = f"""
    YOUR EXTRACTION PROMPT HERE
    
    Ticket: {text}
    """
    
    try:
        response = model.generate_content(extraction_prompt)
        data = json.loads(response.text.strip())
        return TicketAnalysis(**data)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print(f"Raw response: {response.text[:200]}")
        return None

for t in tickets[:3]:
    result = extract_ticket_info(t["text"])
    print(f"Ticket {t['id']}: {result}")

**Analysis questions:**

1. At roughly 200 input tokens + 50 output tokens per ticket, and `gemini-2.0-flash` paid-tier pricing ($0.10 per 1M input tokens, $0.40 per 1M output tokens), what is the approximate daily API cost for 10,000 tickets? How does that compare to the free tier limit?
2. Under what conditions would few-shot reliably outperform zero-shot for classification tasks like this?
3. In a production pipeline, beyond `JSONDecodeError`, what other error scenarios should you handle?

*(Write your answers here.)*